# ⚡ XFarmaCare — Motor de Ofertas Automáticas
## Reglas de Negocio que Activan Intervenciones sin Humanos

**Objetivo:** Cruzar (Probabilidad de Churn × Índice de Priorización × Segmento)
para detonar acciones automáticas. Cada cliente recibe la oferta que maximiza
el retorno sobre la inversión en retención.

**Inputs (todos conectados por `customer_id`):**
- `output_churn_scores.csv` → probabilidad y riesgo de churn
- `output_segmentos_clientes.csv` → cluster y segmento conductual
- `output_indice_priorizacion.csv` → score de valor del cliente (generado en R)
- `output_uplift_scores.csv` → si es "persuadible" (generado en R)

**Output:**
- `output_motor_ofertas.csv` → acción recomendada por cliente
---

In [ ]:
# === Setup ===
import pandas as pd
import numpy as np
import os, warnings
from datetime import datetime
warnings.filterwarnings('ignore')

OUTPUT_DIR = "outputs/"
DATA_DIR = "data/"

print(f"[{datetime.now()}] Iniciando Motor de Ofertas Automáticas...")

In [ ]:
# === Cargar todos los inputs del modelo estrella ===
churn = pd.read_csv(f"{OUTPUT_DIR}output_churn_scores.csv")
segmentos = pd.read_csv(f"{OUTPUT_DIR}output_segmentos_clientes.csv")
clientes = pd.read_csv(f"{DATA_DIR}dim_clientes.csv")

# Intentar cargar outputs de R (si ya se ejecutaron)
try:
    prioridad = pd.read_csv(f"{OUTPUT_DIR}output_indice_priorizacion.csv")
    tiene_prioridad = True
    print("Índice de priorización cargado desde R.")
except FileNotFoundError:
    tiene_prioridad = False
    print("AVISO: output_indice_priorizacion.csv no encontrado. Se generará uno básico.")

try:
    uplift = pd.read_csv(f"{OUTPUT_DIR}output_uplift_scores.csv")
    tiene_uplift = True
    print("Scores de uplift cargados desde R.")
except FileNotFoundError:
    tiene_uplift = False
    print("AVISO: output_uplift_scores.csv no encontrado. Se usará churn como proxy.")

# Unir todo
motor = churn[['customer_id', 'probabilidad_churn', 'riesgo_churn']].copy()
motor = motor.merge(segmentos[['customer_id', 'segmento']], on='customer_id', how='left')
motor = motor.merge(clientes[['customer_id', 'es_cronico', 'condicion_cronica',
                               'programa_lealtad', 'nivel_lealtad', 'ingreso_estimado']],
                    on='customer_id', how='left')

if tiene_prioridad:
    motor = motor.merge(prioridad[['customer_id', 'indice_priorizacion', 'categoria_valor']],
                       on='customer_id', how='left')
else:
    # Proxy simple: basado en churn y segmento
    motor['indice_priorizacion'] = motor['probabilidad_churn'] * 100
    motor['categoria_valor'] = 'Sin calcular'

if tiene_uplift:
    motor = motor.merge(uplift[['customer_id', 'uplift_score', 'tipo_respuesta']],
                       on='customer_id', how='left')
else:
    motor['uplift_score'] = 0
    motor['tipo_respuesta'] = 'Sin calcular'

motor['segmento'] = motor['segmento'].fillna('Sin segmentar')
print(f"\nClientes en el motor: {len(motor):,}")
motor.head()

## Reglas de Negocio del Motor
Las reglas combinan riesgo de churn, valor del cliente y tipo de respuesta
esperada (uplift) para decidir qué acción tomar y con qué intensidad.

In [ ]:
# === REGLAS DE NEGOCIO ===

def decidir_accion(row):
    """
    Lógica del motor de ofertas automáticas.
    Cruza churn × valor × segmento × uplift para generar la acción.
    
    Retorna: (accion, descuento_pct, canal_contacto, prioridad, justificacion)
    """
    churn_prob = row['probabilidad_churn']
    segmento = row['segmento']
    es_cronico = row['es_cronico']
    lealtad = row['nivel_lealtad']
    uplift_tipo = row['tipo_respuesta']
    
    # ── REGLA 0: Si el uplift dice "No contactar", no desperdiciar recursos ──
    if uplift_tipo == 'Sleeping Dog':
        return ("No contactar", 0, "Ninguno", "Baja",
                "Cliente que empeora con contacto. No intervenir.")
    
    # ── REGLA 1: Crónico de alto valor con riesgo crítico ──
    if es_cronico and churn_prob >= 0.70 and segmento == "Crónico Alto Valor":
        return ("Descuento + Llamada VIP", 15, "Llamada personalizada", "Urgente",
                "Paciente crónico de alto valor en riesgo crítico. Pérdida de CLV garantizado.")
    
    # ── REGLA 2: Crónico en riesgo (adherencia decayendo) ──
    if es_cronico and churn_prob >= 0.50 and segmento == "Crónico en Riesgo":
        return ("Recordatorio + Descuento receta", 10, "WhatsApp", "Alta",
                "Adherencia decayendo. Intervención preventiva para evitar abandono terapéutico.")
    
    # ── REGLA 3: Cliente frecuente con churn alto ──
    if churn_prob >= 0.60 and segmento == "Frecuente No Crónico":
        return ("Puntos bonus + Oferta personalizada", 8, "App Push", "Alta",
                "Cliente frecuente mostrando señales de fuga. Retener con incentivos de lealtad.")
    
    # ── REGLA 4: Buscador de ofertas con riesgo medio-alto ──
    if segmento == "Buscador de Ofertas" and churn_prob >= 0.40:
        return ("Cupón exclusivo", 12, "Email", "Media",
                "Sensible a precio. Cupón para mantener enganche sin erosionar margen.")
    
    # ── REGLA 5: Digital activo con churn moderado ──
    if segmento == "Digital Activo" and churn_prob >= 0.40:
        return ("Envío gratis + Promo app", 5, "App Push", "Media",
                "Usuario digital. Mantener con beneficios del canal que prefiere.")
    
    # ── REGLA 6: Cualquier crónico con riesgo moderado ──
    if es_cronico and churn_prob >= 0.35:
        return ("Recordatorio de receta + Descuento", 7, "SMS", "Media",
                "Crónico con riesgo moderado. Recordatorio proactivo de reabastecimiento.")
    
    # ── REGLA 7: Churn alto pero bajo valor (esporádico) ──
    if churn_prob >= 0.60 and segmento == "Esporádico / Inactivo":
        return ("Email de reactivación", 5, "Email", "Baja",
                "Cliente esporádico. Esfuerzo mínimo, no justifica inversión alta.")
    
    # ── REGLA 8: Riesgo bajo, cliente satisfecho ──
    if churn_prob < 0.20:
        return ("Mantenimiento — Sin acción", 0, "Ninguno", "Monitoreo",
                "Cliente estable. No gastar en quien ya se queda.")
    
    # ── DEFAULT: Riesgo moderado general ──
    return ("Comunicación estándar de lealtad", 3, "Email", "Baja",
            "Riesgo moderado. Comunicación rutinaria del programa.")

# Aplicar las reglas
resultados = motor.apply(decidir_accion, axis=1, result_type='expand')
resultados.columns = ['accion_recomendada', 'descuento_pct', 'canal_contacto',
                       'prioridad_accion', 'justificacion']

motor = pd.concat([motor, resultados], axis=1)
motor['fecha_generacion'] = datetime.now().strftime('%Y-%m-%d')
motor['costo_estimado_dop'] = (motor['descuento_pct'] / 100 * 500).round(0)  # Proxy

print("Motor de ofertas ejecutado.")
print(f"\nDistribución de acciones:")
print(motor['accion_recomendada'].value_counts())

In [ ]:
# === Resumen ejecutivo ===
print("="*60)
print("RESUMEN EJECUTIVO — MOTOR DE OFERTAS AUTOMÁTICAS")
print("="*60)

print(f"\nTotal clientes evaluados: {len(motor):,}")
print(f"\nPor prioridad de acción:")
for p in ['Urgente', 'Alta', 'Media', 'Baja', 'Monitoreo']:
    n = (motor['prioridad_accion'] == p).sum()
    print(f"  {p:12s}: {n:>5,} clientes ({n/len(motor)*100:.1f}%)")

print(f"\nCosto estimado total de intervención:")
costo_total = motor['costo_estimado_dop'].sum()
print(f"  RD$ {costo_total:,.0f} DOP")

print(f"\nClientes que NO se deben contactar (Sleeping Dogs):")
sd = (motor['accion_recomendada'] == 'No contactar').sum()
print(f"  {sd:,} clientes — ahorro de recursos")

print(f"\nAcciones urgentes (crónicos alto valor en riesgo):")
urgentes = motor[motor['prioridad_accion'] == 'Urgente']
print(f"  {len(urgentes):,} clientes requieren intervención inmediata")

In [ ]:
# === Exportar ===
cols_export = ['customer_id', 'probabilidad_churn', 'riesgo_churn', 'segmento',
               'es_cronico', 'accion_recomendada', 'descuento_pct',
               'canal_contacto', 'prioridad_accion', 'justificacion',
               'costo_estimado_dop', 'fecha_generacion']

motor[cols_export].to_csv(f"{OUTPUT_DIR}output_motor_ofertas.csv", index=False)
print(f"\nMotor exportado: {OUTPUT_DIR}output_motor_ofertas.csv")
print(f"Columnas: {len(cols_export)}")
print(f"Filas: {len(motor):,}")

## ✅ Motor de Ofertas Completado
- Acciones automáticas generadas para toda la cartera
- **Sleeping Dogs** identificados para evitar gasto inútil
- **Crónicos de alto valor** priorizados con intervención inmediata
- Todo conectado por `customer_id`

**Nota:** Este motor se enriquece cuando se ejecutan los scripts de R:
- `05_indice_priorizacion.R` → agrega el score de valor real
- `06_uplift_modelado_causal.R` → identifica persuadibles vs sleeping dogs